# Garden Beds — Data Cleaning 

**Source:** `renewals-for-nature-city-of-melbourne-garden-bed-inventory-2024.csv`  
**Output:** `gardenbeds_cleaned.csv`



## 1 — Setup

In [1]:
from pathlib import Path
import pandas as pd

# keep only columns useful for first-version garden bed tasks
GARDEN_KEEP_COLS = [
    "ObjectID",
    "Asset ID",
    "Assessed bed type",
    "Bed Type Origin",
    "Botanical name",
    "Common name",
    "Origin",
    "Form",
    "Level of certainty",
    "Plant condition",
    "Overall Condition Rating",
    "Neighbourhood",
    "Area (m2)",
    "Site Type",
    "Site",
    "Irrigation",
    "Corridor",
    "X_Coord",
    "Y_Coord",
]

# Rename to snake_case
GARDEN_RENAME_MAP = {
    "ObjectID": "object_id",
    "Asset ID": "asset_id",
    "Assessed bed type": "assessed_bed_type",
    "Bed Type Origin": "bed_type_origin",
    "Botanical name": "botanical_name",
    "Common name": "common_name",
    "Origin": "origin",
    "Form": "form",
    "Level of certainty": "level_of_certainty",
    "Plant condition": "plant_condition",
    "Overall Condition Rating": "overall_condition_rating",
    "Neighbourhood": "neighbourhood",
    "Area (m2)": "area_m2",
    "Site Type": "site_type",
    "Site": "site",
    "Irrigation": "irrigation",
    "Corridor": "corridor",
    "X_Coord": "x_coord",
    "Y_Coord": "y_coord",
}

def load_gardenbed_data(filepath: Path) -> pd.DataFrame:
    """
    Read raw garden bed CSV, keep useful columns, rename to snake_case.
    """
    df = pd.read_csv(filepath, encoding="utf-8-sig", low_memory=False)

    print("Raw shape:", df.shape)
    print("\nRaw columns:")
    print(df.columns.tolist())

    df = df[GARDEN_KEEP_COLS].copy()
    df = df.rename(columns=GARDEN_RENAME_MAP)

    print("\nAfter Step 1 shape:", df.shape)
    print("\nAfter Step 1 columns:")
    print(df.columns.tolist())

    return df



In [2]:
garden_path = Path("renewals-for-nature-city-of-melbourne-garden-bed-inventory-2024.csv")

garden_raw = load_gardenbed_data(garden_path)

garden_raw.head()

Raw shape: (17721, 39)

Raw columns:
['ObjectID', 'Asset ID no fill', 'Asset ID', 'Asessment date (current)', 'Assessed bed type', 'Bed Type Origin', 'Spatial dimensions match', 'Botanical name', 'Common name', 'Origin', 'Form', 'Level of certainty', 'Dominant species only above >25%', 'Logs Present/absent', 'Logs % cover lineal m', 'Rocks Present/absent', 'Rocks Percentage cover', 'Plant cover (ground layer)', 'Plant cover score', 'Plant condition', 'Plant Condition Score', 'Mulch Condition', 'Mulch score', 'Weed Percentage number', 'Weed cover score', 'Infrastructure Condition', 'Infrastructure score', 'TOTAL SCORE /26 Beds - /15 Mulch', 'Overall Condition Rating', 'Condition rating score', 'Neighbourhood', 'Area (m2)', 'Site Type', 'Site', 'Irrigation', 'Previous Condition', 'Corridor', 'X_Coord', 'Y_Coord']

After Step 1 shape: (17721, 19)

After Step 1 columns:
['object_id', 'asset_id', 'assessed_bed_type', 'bed_type_origin', 'botanical_name', 'common_name', 'origin', 'form', 'lev

,object_id,asset_id,assessed_bed_type,bed_type_origin,botanical_name,common_name,origin,form,level_of_certainty,plant_condition,overall_condition_rating,neighbourhood,area_m2,site_type,site,irrigation,corridor,x_coord,y_coord
0,12704,1531795,Bioretention bed,Native,Dianella caerulea,Paroo Lily,Native,Grasslike Perennial,1.0,Mature & healthy,Poor condition,Docklands,8,Park,Yarras Edge,No,1531795,NaN,NaN
1,12720,1531796,Perennial bed,Native,NaN,NaN,NaN,NaN,NaN,Aged (may be established plants but near end o...,Total failure,Docklands,2,Park,Yarras Edge,No,1531796,318466.2480,5811902.389
2,12299,1531798,Perennial bed,Mixed,NaN,NaN,NaN,NaN,NaN,Mature & healthy,Good condition,Docklands,24,Park,Yarras Edge,No,1531798,318494.6602,5811915.095
3,12301,1531798,Perennial bed,Mixed,Leucadendron salignum,Common Sunshine Conebush,Exotic,Shrub,1.0,Mature & healthy,Good condition,Docklands,24,Park,Yarras Edge,No,1531798,NaN,NaN
4,12637,1531801,Perennial bed,Exotic,Nandina domestica,Sacred Bamboo,Exotic,Shrub,1.0,"Poor (e.g. diseased, damaged, nutrient deficient)",Total failure,Docklands,2,Park,Yarras Edge,No,1531801,NaN,NaN


## 2  Raw Inspection



In [3]:
# inspect missingness of key attributes

garden_key_cols = [
    "asset_id",
    "assessed_bed_type",
    "botanical_name",
    "common_name",
    "neighbourhood",
    "site_type",
    "site",
    "x_coord",
    "y_coord",
]

missing_summary = pd.DataFrame({
    "missing_count": garden_raw[garden_key_cols].isna().sum(),
    "missing_ratio": garden_raw[garden_key_cols].isna().mean().round(4)
}).sort_values("missing_ratio", ascending=False)

missing_summary

,missing_count,missing_ratio
x_coord,13696,0.7729
y_coord,13696,0.7729
common_name,4873,0.2750
botanical_name,4088,0.2307
neighbourhood,649,0.0366
site_type,649,0.0366
site,649,0.0366
asset_id,0,0.0000
assessed_bed_type,0,0.0000


In [4]:
# Check how many rows have usable place / name / coordinates

has_xy = garden_raw["x_coord"].notna() & garden_raw["y_coord"].notna()
has_place = (
    garden_raw["site"].notna()
    | garden_raw["site_type"].notna()
    | garden_raw["neighbourhood"].notna()
)
has_name = garden_raw["common_name"].notna() | garden_raw["botanical_name"].notna()
has_bed_type = garden_raw["assessed_bed_type"].notna()

coverage_summary = pd.DataFrame({
    "condition": [
        "has_xy",
        "has_place",
        "has_name",
        "has_bed_type",
        "has_xy_and_place",
        "has_xy_and_name",
        "has_place_and_name",
        "has_xy_place_name",
    ],
    "count": [
        has_xy.sum(),
        has_place.sum(),
        has_name.sum(),
        has_bed_type.sum(),
        (has_xy & has_place).sum(),
        (has_xy & has_name).sum(),
        (has_place & has_name).sum(),
        (has_xy & has_place & has_name).sum(),
    ]
})

coverage_summary["ratio"] = (coverage_summary["count"] / len(garden_raw)).round(4)
coverage_summary

,condition,count,ratio
0,has_xy,4025,0.2271
1,has_place,17072,0.9634
2,has_name,13634,0.7694
3,has_bed_type,17721,1.0000
4,has_xy_and_place,3933,0.2219
5,has_xy_and_name,1,0.0001
6,has_place_and_name,13079,0.7381
7,has_xy_place_name,1,0.0001


## 3  Clean



In [5]:
GARDEN_TEXT_COLS = [
    "assessed_bed_type",
    "bed_type_origin",
    "botanical_name",
    "common_name",
    "origin",
    "form",
    "plant_condition",
    "overall_condition_rating",
    "neighbourhood",
    "site_type",
    "site",
    "irrigation",
]

GARDEN_PLACE_COLS = [
    "site",
    "site_type",
    "neighbourhood",
]

def normalize_text(value):
    """
    text normalization:
    - keep missing as NA
    - strip spaces
    - collapse repeated spaces
    """
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()
    if value == "":
        return pd.NA

    value = " ".join(value.split())
    return value


def normalize_place_value(value):
    """
    Normalize place-related values more strictly.
    """
    value = normalize_text(value)

    if pd.isna(value):
        return pd.NA

    low = str(value).lower()
    if low in {"missing", "nan", "none", "null", "0"}:
        return pd.NA

    return value


def clean_gardenbed_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean raw garden bed data, but do not convert coordinates yet.
    """
    clean_df = df.copy()

    #  text normalization
    for col in GARDEN_TEXT_COLS:
        clean_df[col] = clean_df[col].apply(normalize_text)

    #  stricter place normalization
    for col in GARDEN_PLACE_COLS:
        clean_df[col] = clean_df[col].apply(normalize_place_value)

    #  numeric conversion
    clean_df["object_id"] = pd.to_numeric(clean_df["object_id"], errors="coerce").astype("Int64")
    clean_df["asset_id"] = pd.to_numeric(clean_df["asset_id"], errors="coerce").astype("Int64")
    clean_df["level_of_certainty"] = pd.to_numeric(clean_df["level_of_certainty"], errors="coerce")
    clean_df["area_m2"] = pd.to_numeric(clean_df["area_m2"], errors="coerce")
    clean_df["x_coord"] = pd.to_numeric(clean_df["x_coord"], errors="coerce")
    clean_df["y_coord"] = pd.to_numeric(clean_df["y_coord"], errors="coerce")

    # remove exact duplicate identifier pairs
    clean_df = clean_df.drop_duplicates(subset=["object_id", "asset_id"], keep="first").copy()

    return clean_df


def build_garden_scene_candidates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep first-version scene-based candidates only:
    rows with usable coordinates and usable place context.
    """
    out = df.copy()

    has_xy = out["x_coord"].notna() & out["y_coord"].notna()
    has_place = out["site"].notna() | out["site_type"].notna() | out["neighbourhood"].notna()

    out = out[has_xy & has_place].copy()

    return out



In [6]:
garden_clean = clean_gardenbed_data(garden_raw)
garden_scene = build_garden_scene_candidates(garden_clean)

print("garden_raw shape   :", garden_raw.shape)
print("garden_clean shape :", garden_clean.shape)
print("garden_scene shape :", garden_scene.shape)

garden_scene.head()

garden_raw shape   : (17721, 19)
garden_clean shape : (17721, 19)
garden_scene shape : (3931, 19)


,object_id,asset_id,assessed_bed_type,bed_type_origin,botanical_name,common_name,origin,form,level_of_certainty,plant_condition,overall_condition_rating,neighbourhood,area_m2,site_type,site,irrigation,corridor,x_coord,y_coord
1,12720,1531796,Perennial bed,Native,<NA>,<NA>,<NA>,<NA>,NaN,Aged (may be established plants but near end o...,Total failure,Docklands,2.0,Park,Yarras Edge,No,1531796,318466.2480,5811902.389
2,12299,1531798,Perennial bed,Mixed,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,Good condition,Docklands,24.0,Park,Yarras Edge,No,1531798,318494.6602,5811915.095
5,12645,1531802,Perennial bed,Mixed,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,Good condition,Docklands,7.0,Park,Yarras Edge,No,1531802,318585.4162,5811882.757
10,12638,1531807,Perennial bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,"Poor (e.g. diseased, damaged, nutrient deficient)",Total failure,Docklands,2.0,Park,Yarras Edge,No,1531807,318587.6399,5811870.542
15,9065,1531831,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,Good condition,CBD Hoddle Grid,6.0,Street,Swanston Street between Little Bourke Street a...,No,No,320852.9140,5813093.902


In [7]:
print("\nMissing values in garden_scene:")
print(garden_scene[[
    "assessed_bed_type",
    "site",
    "site_type",
    "neighbourhood",
    "x_coord",
    "y_coord"
]].isna().sum())

print("\nTop 15 assessed_bed_type in garden_scene:")
print(garden_scene["assessed_bed_type"].value_counts(dropna=False).head(15))

print("\nTop 15 site values in garden_scene:")
print(garden_scene["site"].value_counts(dropna=False).head(15))

print("\nTop 15 site_type values in garden_scene:")
print(garden_scene["site_type"].value_counts(dropna=False).head(15))


Missing values in garden_scene:
assessed_bed_type     0
site                  0
site_type             0
neighbourhood        38
x_coord               0
y_coord               0
dtype: int64

Top 15 assessed_bed_type in garden_scene:
assessed_bed_type
Perennial bed       2644
Mulch bed            696
Hedge                186
Groundcover bed      177
Bioretention bed     141
Annual bed            58
Bulb understorey      24
Unknown                4
Bulb meadow            1
Name: count, dtype: int64

Top 15 site values in garden_scene:
site
Royal Park                504
Fitzroy Gardens           158
Carlton Gardens South     158
Fawkner Park               90
Yarras Edge                87
Queen Victoria Gardens     73
New Quay Promenade         72
Kings Domain South         66
Kings Domain               59
Docklands Park             58
Riverside Park             53
Vincent Place Park         48
Flagstaff Gardens          46
Victoria Green             44
Warun Biik Park            43
Name: 

## 4 Building attribute

In [8]:
print("Top 30 raw site values:")
print(garden_scene["site"].value_counts(dropna=False).head(30))

print("\nRows where site looks like placeholder:")
placeholder_mask = garden_scene["site"].astype(str).str.strip().isin(["...", ".", "-", "na", "n/a", "unknown"])
print(garden_scene.loc[placeholder_mask, ["asset_id", "site", "site_type", "neighbourhood"]].head(20))
print("Placeholder-like site rows:", placeholder_mask.sum())

Top 30 raw site values:
site
Royal Park                                                    504
Fitzroy Gardens                                               158
Carlton Gardens South                                         158
Fawkner Park                                                   90
Yarras Edge                                                    87
Queen Victoria Gardens                                         73
New Quay Promenade                                             72
Kings Domain South                                             66
Kings Domain                                                   59
Docklands Park                                                 58
Riverside Park                                                 53
Vincent Place Park                                             48
Flagstaff Gardens                                              46
Victoria Green                                                 44
Warun Biik Park                                

In [9]:
GARDEN_PLACE_PLACEHOLDERS = {
    "...", ".", "-", "na", "n/a", "unknown", "missing", "null", "none", "0"
}

def normalize_garden_place(value):
    """
    Normalize garden place text and remove placeholder-like values.
    """
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    if value == "":
        return pd.NA

    value = " ".join(value.split())

    if value.lower() in GARDEN_PLACE_PLACEHOLDERS:
        return pd.NA

    return value


def build_garden_location_fields(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build location_group and location_label for garden bed data.
    """
    out = df.copy()

    out["site_norm"] = out["site"].apply(normalize_garden_place)
    out["site_type_norm"] = out["site_type"].apply(normalize_garden_place)
    out["neighbourhood_norm"] = out["neighbourhood"].apply(normalize_garden_place)

    def make_location_group(row):
        site = row["site_norm"]
        site_type = row["site_type_norm"]
        neighbourhood = row["neighbourhood_norm"]

        parts = [p for p in [site, site_type, neighbourhood] if pd.notna(p)]

        if len(parts) == 0:
            return pd.NA

        return " | ".join(parts)

    def make_location_label(row):
        site = row["site_norm"]
        site_type = row["site_type_norm"]
        neighbourhood = row["neighbourhood_norm"]

        if pd.notna(site):
            return site
        elif pd.notna(site_type) and pd.notna(neighbourhood):
            return f"{site_type} | {neighbourhood}"
        elif pd.notna(site_type):
            return site_type
        elif pd.notna(neighbourhood):
            return neighbourhood
        else:
            return pd.NA

    out["location_group"] = out.apply(make_location_group, axis=1)
    out["location_label"] = out.apply(make_location_label, axis=1)

    return out

In [12]:
garden_loc = build_garden_location_fields(garden_scene)

print("Missing location_group:", garden_loc["location_group"].isna().sum())
print("Missing location_label:", garden_loc["location_label"].isna().sum())
print("Unique location_group :", garden_loc["location_group"].nunique(dropna=True))
print("Unique location_label :", garden_loc["location_label"].nunique(dropna=True))

garden_loc.head(10)

Missing location_group: 0
Missing location_label: 0
Unique location_group : 534
Unique location_label : 530


,object_id,asset_id,assessed_bed_type,bed_type_origin,botanical_name,common_name,origin,form,level_of_certainty,plant_condition,...,site,irrigation,corridor,x_coord,y_coord,site_norm,site_type_norm,neighbourhood_norm,location_group,location_label
1,12720,1531796,Perennial bed,Native,<NA>,<NA>,<NA>,<NA>,NaN,Aged (may be established plants but near end o...,...,Yarras Edge,No,1531796,318466.2480,5811902.389,Yarras Edge,Park,Docklands,Yarras Edge | Park | Docklands,Yarras Edge
2,12299,1531798,Perennial bed,Mixed,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,Yarras Edge,No,1531798,318494.6602,5811915.095,Yarras Edge,Park,Docklands,Yarras Edge | Park | Docklands,Yarras Edge
5,12645,1531802,Perennial bed,Mixed,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,Yarras Edge,No,1531802,318585.4162,5811882.757,Yarras Edge,Park,Docklands,Yarras Edge | Park | Docklands,Yarras Edge
10,12638,1531807,Perennial bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,"Poor (e.g. diseased, damaged, nutrient deficient)",...,Yarras Edge,No,1531807,318587.6399,5811870.542,Yarras Edge,Park,Docklands,Yarras Edge | Park | Docklands,Yarras Edge
15,9065,1531831,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,Swanston Street between Little Bourke Street a...,No,No,320852.9140,5813093.902,Swanston Street between Little Bourke Street a...,Street,CBD Hoddle Grid,Swanston Street between Little Bourke Street a...,Swanston Street between Little Bourke Street a...
18,9147,1531834,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,Swanston Street between Little Bourke Street a...,No,No,320864.5440,5813113.112,Swanston Street between Little Bourke Street a...,Street,CBD Hoddle Grid,Swanston Street between Little Bourke Street a...,Swanston Street between Little Bourke Street a...
20,9143,1531838,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,Swanston Street between Lonsdale Street and Li...,No,No,320844.4038,5813165.612,Swanston Street between Lonsdale Street and Li...,Street,CBD Hoddle Grid,Swanston Street between Lonsdale Street and Li...,Swanston Street between Lonsdale Street and Li...
21,9142,1531840,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,Swanston Street between Lonsdale Street and Li...,No,No,320839.2438,5813178.502,Swanston Street between Lonsdale Street and Li...,Street,CBD Hoddle Grid,Swanston Street between Lonsdale Street and Li...,Swanston Street between Lonsdale Street and Li...
22,9019,1531841,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,Swanston Street between Collins Street and Lit...,No,No,320986.5145,5812749.422,Swanston Street between Collins Street and Lit...,Street,CBD Hoddle Grid,Swanston Street between Collins Street and Lit...,Swanston Street between Collins Street and Lit...
25,9067,1531843,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,Swanston Street between Little Bourke Street a...,No,No,320847.7539,5813106.512,Swanston Street between Little Bourke Street a...,Street,CBD Hoddle Grid,Swanston Street between Little Bourke Street a...,Swanston Street between Little Bourke Street a...


In [11]:
# convert the lat and lon to standard form

from pyproj import Transformer

# convert GDA2020 / MGA Zone 55 to WGS84 lat/lon
garden_transformer = Transformer.from_crs("EPSG:7855", "EPSG:4326", always_xy=True)

def convert_garden_xy_to_latlon(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert projected x_coord / y_coord to longitude / latitude.
    """
    out = df.copy()

    def project_one_row(row):
        x = row["x_coord"]
        y = row["y_coord"]

        if pd.isna(x) or pd.isna(y):
            return pd.Series({"longitude": pd.NA, "latitude": pd.NA})

        lon, lat = garden_transformer.transform(float(x), float(y))
        return pd.Series({"longitude": lon, "latitude": lat})

    coords = out.apply(project_one_row, axis=1)
    out = pd.concat([out, coords], axis=1)

    return out

garden_geo = convert_garden_xy_to_latlon(garden_loc)

print("Missing longitude:", garden_geo["longitude"].isna().sum())
print("Missing latitude :", garden_geo["latitude"].isna().sum())

print("\nCoordinate summary:")
print(garden_geo[["latitude", "longitude"]].describe())

garden_geo.head(10)

Missing longitude: 0
Missing latitude : 0

Coordinate summary:
          latitude    longitude
count  3931.000000  3931.000000
mean    -37.808260   144.952144
std       0.015210     0.018899
min     -37.845331   144.900632
25%     -37.821591   144.939642
50%     -37.810039   144.950438
75%     -37.795344   144.969974
max     -37.775675   144.991022


,object_id,asset_id,assessed_bed_type,bed_type_origin,botanical_name,common_name,origin,form,level_of_certainty,plant_condition,...,corridor,x_coord,y_coord,site_norm,site_type_norm,neighbourhood_norm,location_group,location_label,longitude,latitude
1,12720,1531796,Perennial bed,Native,<NA>,<NA>,<NA>,<NA>,NaN,Aged (may be established plants but near end o...,...,1531796,318466.2480,5811902.389,Yarras Edge,Park,Docklands,Yarras Edge | Park | Docklands,Yarras Edge,144.937435,-37.822249
2,12299,1531798,Perennial bed,Mixed,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,1531798,318494.6602,5811915.095,Yarras Edge,Park,Docklands,Yarras Edge | Park | Docklands,Yarras Edge,144.937761,-37.822140
5,12645,1531802,Perennial bed,Mixed,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,1531802,318585.4162,5811882.757,Yarras Edge,Park,Docklands,Yarras Edge | Park | Docklands,Yarras Edge,144.938783,-37.822449
10,12638,1531807,Perennial bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,"Poor (e.g. diseased, damaged, nutrient deficient)",...,1531807,318587.6399,5811870.542,Yarras Edge,Park,Docklands,Yarras Edge | Park | Docklands,Yarras Edge,144.938805,-37.822560
15,9065,1531831,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,No,320852.9140,5813093.902,Swanston Street between Little Bourke Street a...,Street,CBD Hoddle Grid,Swanston Street between Little Bourke Street a...,Swanston Street between Little Bourke Street a...,144.964831,-37.811988
18,9147,1531834,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,No,320864.5440,5813113.112,Swanston Street between Little Bourke Street a...,Street,CBD Hoddle Grid,Swanston Street between Little Bourke Street a...,Swanston Street between Little Bourke Street a...,144.964968,-37.811817
20,9143,1531838,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,No,320844.4038,5813165.612,Swanston Street between Lonsdale Street and Li...,Street,CBD Hoddle Grid,Swanston Street between Lonsdale Street and Li...,Swanston Street between Lonsdale Street and Li...,144.964752,-37.811340
21,9142,1531840,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,No,320839.2438,5813178.502,Swanston Street between Lonsdale Street and Li...,Street,CBD Hoddle Grid,Swanston Street between Lonsdale Street and Li...,Swanston Street between Lonsdale Street and Li...,144.964696,-37.811223
22,9019,1531841,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,No,320986.5145,5812749.422,Swanston Street between Collins Street and Lit...,Street,CBD Hoddle Grid,Swanston Street between Collins Street and Lit...,Swanston Street between Collins Street and Lit...,144.966262,-37.815117
25,9067,1531843,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,No,320847.7539,5813106.512,Swanston Street between Little Bourke Street a...,Street,CBD Hoddle Grid,Swanston Street between Little Bourke Street a...,Swanston Street between Little Bourke Street a...,144.964775,-37.811873


## 5 task build


In [13]:



def normalize_bed_type_for_sentence(value):
    """
    Convert bed type to sentence-friendly text.
    """
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()
    if value == "":
        return pd.NA

    return value.lower()

def make_criteria(x):
    return f"{x.capitalize()} should be present in the image."

def build_garden_scene_task_content(
    df: pd.DataFrame,
    series_id: int = 1,
    geofence_radius_meter: int = 50,
    base_difficulty: int = 1,
    reward_point: int = 10,
) -> pd.DataFrame:
    out = df.copy()

    out = out[out["assessed_bed_type"].notna()].copy()
    out = out[out["location_label"].notna()].copy()
    out = out[out["assessed_bed_type"].str.strip().str.lower() != "unknown"].copy()

    out["bed_type_clean"] = out["assessed_bed_type"].astype(str).str.strip()
    out["bed_type_phrase"] = out["bed_type_clean"].str.lower()

    out["task_name"] = out["bed_type_clean"]

    out["task_description"] = out.apply(
        lambda row: f"Take a picture of {row['bed_type_phrase']} near {row['location_label']}.",
        axis=1
    )

    out["evaluation_criteria"] = out["bed_type_phrase"].apply(
        make_criteria
    )

    out["series_id"] = series_id
    out["environment_type"] = "outdoor"
    out["task_type"] = "photo"
    out["geofence_radius_meter"] = geofence_radius_meter
    out["base_difficulty"] = base_difficulty
    out["reward_point"] = reward_point
    out["is_active"] = True

    return out

garden_task_base = build_garden_scene_task_content(garden_geo)

print("garden_geo shape      :", garden_geo.shape)
print("garden_task_base shape:", garden_task_base.shape)

garden_task_base.head(10)

garden_geo shape      : (3931, 26)
garden_task_base shape: (3927, 38)


,object_id,asset_id,assessed_bed_type,bed_type_origin,botanical_name,common_name,origin,form,level_of_certainty,plant_condition,...,task_name,task_description,evaluation_criteria,series_id,environment_type,task_type,geofence_radius_meter,base_difficulty,reward_point,is_active
1,12720,1531796,Perennial bed,Native,<NA>,<NA>,<NA>,<NA>,NaN,Aged (may be established plants but near end o...,...,Perennial bed,Take a picture of perennial bed near Yarras Edge.,Perennial bed should be present in the image.,1,outdoor,photo,50,1,10,True
2,12299,1531798,Perennial bed,Mixed,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,Perennial bed,Take a picture of perennial bed near Yarras Edge.,Perennial bed should be present in the image.,1,outdoor,photo,50,1,10,True
5,12645,1531802,Perennial bed,Mixed,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,Perennial bed,Take a picture of perennial bed near Yarras Edge.,Perennial bed should be present in the image.,1,outdoor,photo,50,1,10,True
10,12638,1531807,Perennial bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,"Poor (e.g. diseased, damaged, nutrient deficient)",...,Perennial bed,Take a picture of perennial bed near Yarras Edge.,Perennial bed should be present in the image.,1,outdoor,photo,50,1,10,True
15,9065,1531831,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,Groundcover bed,Take a picture of groundcover bed near Swansto...,Groundcover bed should be present in the image.,1,outdoor,photo,50,1,10,True
18,9147,1531834,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,Groundcover bed,Take a picture of groundcover bed near Swansto...,Groundcover bed should be present in the image.,1,outdoor,photo,50,1,10,True
20,9143,1531838,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,Groundcover bed,Take a picture of groundcover bed near Swansto...,Groundcover bed should be present in the image.,1,outdoor,photo,50,1,10,True
21,9142,1531840,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,Groundcover bed,Take a picture of groundcover bed near Swansto...,Groundcover bed should be present in the image.,1,outdoor,photo,50,1,10,True
22,9019,1531841,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,Groundcover bed,Take a picture of groundcover bed near Swansto...,Groundcover bed should be present in the image.,1,outdoor,photo,50,1,10,True
25,9067,1531843,Groundcover bed,Exotic,<NA>,<NA>,<NA>,<NA>,NaN,Mature & healthy,...,Groundcover bed,Take a picture of groundcover bed near Swansto...,Groundcover bed should be present in the image.,1,outdoor,photo,50,1,10,True


In [ ]:
# check task_name
print(garden_task_base["task_name"].value_counts(dropna=False).head(30))

# delete bed from task name
garden_task_base["task_name"] = (
    garden_task_base["task_name"]
    .astype("string")
    .str.replace(r"\s+bed\b", "", regex=True, case=False)
    .str.strip()
)

print(garden_task_base["task_name"].value_counts(dropna=False).head(20))




task_name
Perennial bed       2644
Mulch bed            696
Hedge                186
Groundcover bed      177
Bioretention bed     141
Annual bed            58
Bulb understorey      24
Bulb meadow            1
Name: count, dtype: int64
task_name
Perennial           2644
Mulch                696
Hedge                186
Groundcover          177
Bioretention         141
Annual                58
Bulb understorey      24
Bulb meadow            1
Name: count, dtype: Int64


In [31]:
invalid_task_names = {
    "unknown", "-", "--", "...", ".", "na", "n/a", "none", "null", ""
}

garden_task_base = garden_task_base[
    garden_task_base["task_name"].notna()
].copy()

garden_task_base["task_name"] = garden_task_base["task_name"].astype("string").str.strip()

garden_task_base = garden_task_base[
    ~garden_task_base["task_name"].str.lower().isin(invalid_task_names)
].copy()

garden_task_base = garden_task_base[
    ~garden_task_base["task_name"].str.fullmatch(r"[-.\s]+", na=False)
].copy()

print("After invalid task_name filtering:", len(garden_task_base))
print(garden_task_base["task_name"].value_counts(dropna=False).head(20))



After invalid task_name filtering: 3927
task_name
Perennial           2644
Mulch                696
Hedge                186
Groundcover          177
Bioretention         141
Annual                58
Bulb understorey      24
Bulb meadow            1
Name: count, dtype: Int64


In [32]:


def build_garden_selection_keys(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build internal keys for quantity control on garden bed tasks.
    """
    out = df.copy()

    # bed type key: same scene target
    out["bed_type_key"] = (
        out["assessed_bed_type"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    # location key: prefer the more complete grouping key
    out["location_key"] = (
        out["location_group"]
        .fillna(out["location_label"])
        .astype("string")
        .str.strip()
    )

    # same bed type in same location
    out["bed_location_key"] = out["bed_type_key"] + " || " + out["location_key"]

    return out


def score_garden_tasks(df: pd.DataFrame) -> pd.DataFrame:
    """
    Assign a simple quality score so higher-quality tasks are kept first.
    """
    out = df.copy()
    out["quality_score"] = 0.0

    # 1) specific site is very valuable for user guidance
    out["quality_score"] += out["site"].notna().astype(int) * 3

    # 2) site_type and neighbourhood make the place more usable
    out["quality_score"] += out["site_type"].notna().astype(int) * 1
    out["quality_score"] += out["neighbourhood"].notna().astype(int) * 1

    # 3) park-like places are usually easier for users to access and complete
    park_like = out["site_type"].astype("string").str.lower().isin(["park", "garden"])
    out["quality_score"] += park_like.astype(int) * 2

    # 4) larger area is usually easier to find and photograph
    out["quality_score"] += out["area_m2"].fillna(0).clip(upper=100) / 25.0

    # 5) certainty is useful when present
    if "level_of_certainty" in out.columns:
        certainty = pd.to_numeric(out["level_of_certainty"], errors="coerce").fillna(0)
        out["quality_score"] += certainty.clip(lower=0, upper=3) * 0.5

    # 6) healthier / better-condition beds preferred
    plant_condition_text = out["plant_condition"].astype("string").str.lower()
    overall_condition_text = out["overall_condition_rating"].astype("string").str.lower()

    healthy_like = plant_condition_text.str.contains(
        "healthy|mature & healthy|good", case=False, na=False
    )
    poor_like = overall_condition_text.str.contains(
        "total failure|poor condition", case=False, na=False
    )

    out["quality_score"] += healthy_like.astype(int) * 1.5
    out["quality_score"] -= poor_like.astype(int) * 1.5

    return out


def thin_garden_tasks(
    df: pd.DataFrame,
    max_per_bed_location: int = 1,
    max_per_bed_type_global: int = 40,
    max_per_location_global: int = 8,
) -> pd.DataFrame:
    """
    Reduce garden bed task volume with three layers:
    1. same bed type in same location
    2. same bed type globally
    3. same location globally
    """
    out = df.copy()

    # Sort highest-quality rows first
    sort_cols = ["quality_score"]
    ascending = [False]

    if "area_m2" in out.columns:
        sort_cols.append("area_m2")
        ascending.append(False)

    out = out.sort_values(
        by=sort_cols,
        ascending=ascending,
        na_position="last"
    ).copy()

    # 1) same bed type in same location
    out = out.groupby(
        "bed_location_key",
        as_index=False,
        group_keys=False
    ).head(max_per_bed_location)

    # 2) same bed type globally
    out = out.groupby(
        "bed_type_key",
        as_index=False,
        group_keys=False
    ).head(max_per_bed_type_global)

    # 3) same location globally
    out = out.groupby(
        "location_key",
        as_index=False,
        group_keys=False
    ).head(max_per_location_global)

    return out



In [33]:
garden_ranked = build_garden_selection_keys(garden_task_base)
garden_ranked = score_garden_tasks(garden_ranked)

garden_selected = thin_garden_tasks(
    garden_ranked,
    max_per_bed_location=1,
    max_per_bed_type_global=40,
    max_per_location_global=8,
)


## 6 Determination of final cols

In [34]:
FINAL_TASK_COLS = [
    "series_id",
    "task_name",
    "task_description",
    "evaluation_criteria",
    "environment_type",
    "task_type",
    "location_label",
    "latitude",
    "longitude",
    "st_point",
    "geofence_radius_meter",
    "base_difficulty",
    "reward_point",
    "is_active",
]

def build_st_point(longitude, latitude):
    """
    Build SQL-ready WKT point text: POINT(longitude latitude)
    """
    if pd.isna(longitude) or pd.isna(latitude):
        return pd.NA
    return f"POINT({longitude} {latitude})"


def build_garden_scene_task_seed(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build final garden bed task seed aligned to the agreed export schema.
    """
    out = df.copy()

    out["st_point"] = out.apply(lambda row: 
                                build_st_point(row["longitude"], row["latitude"]), axis=1)
    
    # keep only the neccessary cols
    out = out[FINAL_TASK_COLS].copy()

    return out

flower_task_seed = build_garden_scene_task_seed(garden_selected)
flower_task_seed.head(5)

,series_id,task_name,task_description,evaluation_criteria,environment_type,task_type,location_label,latitude,longitude,st_point,geofence_radius_meter,base_difficulty,reward_point,is_active
2004,1,Perennial,Take a picture of perennial bed near Royal Park.,Perennial bed should be present in the image.,outdoor,photo,Royal Park,-37.786164,144.956762,POINT(144.95676168435145 -37.78616439896781),50,1,10,True
14306,1,Perennial,Take a picture of perennial bed near Kings Dom...,Perennial bed should be present in the image.,outdoor,photo,Kings Domain,-37.826219,144.975466,POINT(144.975466491667 -37.826218789897624),50,1,10,True
3464,1,Perennial,Take a picture of perennial bed near Birrarung...,Perennial bed should be present in the image.,outdoor,photo,Birrarung Marr,-37.819253,144.974879,POINT(144.97487910253946 -37.81925290442416),50,1,10,True
10148,1,Groundcover,Take a picture of groundcover bed near Riversi...,Groundcover bed should be present in the image.,outdoor,photo,Riverside Park,-37.795407,144.915537,POINT(144.91553692710656 -37.79540690730548),50,1,10,True
2446,1,Perennial,Take a picture of perennial bed near CSL Limit...,Perennial bed should be present in the image.,outdoor,photo,CSL Limited 39-79 Poplar Road PARKVILLE VIC 3052,-37.780229,144.942447,POINT(144.9424467950495 -37.78022919340789),50,1,10,True


## 6  Export 



In [35]:
output_path = Path("flower_task_seed.csv")

flower_task_seed.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Exported to: {output_path}")
print(f"Row count: {len(flower_task_seed)}")
print(f"Column count: {len(flower_task_seed.columns)}")

Exported to: flower_task_seed.csv
Row count: 220
Column count: 14


In [36]:
import pandas as pd

input_file = "flower_task_seed.csv"
output_file = "flower_task1.csv"

df = pd.read_csv(input_file)

df["is_active"] = df["is_active"].map({
    True: 1,
    False: 0,
    "True": 1,
    "False": 0,
    "true": 1,
    "false": 0
})

df.to_csv(output_file, index=False, encoding="utf-8-sig")